In [15]:
import os
import glob
import itertools
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.signal import chirp, find_peaks, peak_widths

functions

In [16]:
# 定义函数


def separate_session(df, threshold=3):
    df = pd.read_csv(df, index_col=0)
    df['index_col'] = pd.to_numeric(df.index, errors='coerce')
    check_points = [0]  # 初始化第一个分割点为0（DataFrame的起始位置）
    frame_list = df['index_col'].values.tolist()
    

    # 判断两行之间的时间差，大于threshold则视为新的session
    for i in range(len(frame_list) - 1):
        if (frame_list[i+1] - frame_list[i]) > threshold:
            check_points.append(frame_list[i+1])
    
    check_points.append(frame_list[-1] + 1)  # 添加最后一个分割点

    session_dfs = []
    for i in range(len(check_points) - 1):
        session_df = df[(df['index_col'] >= check_points[i]) & (df['index_col'] < check_points[i+1])]
        session_df.drop(columns='index_col', inplace=True)
        session_dfs.append(session_df)

    return session_dfs
   
 

# 处理分离后的session文件，使每个session的开始时间为0
def process_frame(df):
    # 获取DataFrame的index并转换为float类型的list
    index_list = [float(index) for index in df.index]

    # 判断第一个index的值是否为0，并进行相应处理
    if index_list[0] != 0:
        # 如果第一个index的值不为0，将所有index值减去第一个index值，以实现“相对于第一个index的增量”效果
        index_list = [index - index_list[0] for index in index_list]

    return index_list

  
def process_event_interval(df):
    # 从DataFrame中提取'From Second'和'To Second'列，并转换为二维列表
    event_list = df[['From Second', 'To Second']].values.tolist()

    # 返回处理后的新事件列表
    return event_list



# 获得事件标记
def get_event_label(event_df):
    
    '''
    Aim:
        for OFT
        translate 'event' in eventFile to integer label
        each row (corresponding to each time interval) of event is assigned a label
        called in function `add_event_label()`
    '''
    
    # get event status list
    Event_status = event_df['Event'].values.tolist()
    # compact event label list for three sessions
    event_label = []
    for i in Event_status:   
        if i.strip() == 'sniff':
            event_label.append(11)
        elif i.strip() == 'sniffed' :
            event_label.append(12)
        elif i.strip() == 'freezing':
            event_label.append(21)
        elif i.strip() == 'cs':
            event_label.append(22)
        elif i.strip() == 'In_all':
            event_label.append(23)
        else:
            event_label.append(24)

    return(event_label)


# 添加事件标签
def add_event_label(trace_df, event_path):

    '''
    Aim:
        add event_label to each frame
        one frame can have several label (since mouse can in a location and do something)
    
    Retrun:
        trace dataframe with one label column
    '''
    
    frame_list = process_frame(trace_df)
    #get event dataframe/eventIntervals/eventLabel
#     event_df = pd.read_csv(event_path)
    event_df = pd.read_excel(event_path)
    interval_list = process_event_interval(event_df)
    event_label = get_event_label(event_df)

    
    # add label to each frame
    frame_label_list = []
    for frame in frame_list:
        frame_label = str()
        
        for i in range(len(interval_list)): # one frame can be in several interval
            if min(interval_list[i])  <= frame <= max(interval_list[i]) :
                frame_label = frame_label+ str(event_label[i])

        # convert str to int
        if len(frame_label) == 0:
            frame_label = np.nan
        else:
            frame_label = int(frame_label)

        # append frame label to list
        frame_label_list.append(frame_label)
    
    
    # add label list to trace df 
    new_df = trace_df.copy()
    new_df['Frame_Label']  =  frame_label_list
    new_df = new_df.dropna()
    

    return(new_df)


In [17]:
# 检查特定状态下的帧间隔，并返回满足条件的数据子集
def check_frame_interval(df, which_status_to_check_frame):
    
    # set up which_status_to_check_frame
    
    if which_status_to_check_frame == 'sniff':
        which_status_to_check_frame = str(11)
    elif which_status_to_check_frame == 'sniffed':
        which_status_to_check_frame = str(12)
    elif which_status_to_check_frame == 'freezing':
        which_status_to_check_frame = str(21)
    elif which_status_to_check_frame == 'cs':
        which_status_to_check_frame = str(22)
    elif which_status_to_check_frame == 'In_all':
        which_status_to_check_frame = str(23)
    else:
        raise ValueError('Not approved status. choose from: Investigating_RIGHT_SNIFF, Investigating_LEFT_SNIFF, In_right, In_all, In_left')

    #df = add_event_label(df, event_1_path, event_2_path, event_3_path, mice_path)[which_session]
    frame_label_list = df['Frame_Label'].values.tolist()
    
    # to separate status label, eg:20214111
    n = 2
    
    check_frame_label_list = []
    for line in frame_label_list:
        line = str(line)
        line_list= [line[i:i+n] for i in range(0, len(line), n)]
        if which_status_to_check_frame in line_list:
            check_frame_label_list.append(1)
        else:
            check_frame_label_list.append(0)       
            
    
            
    df['status'] = check_frame_label_list
    df_keep = df[ (df['status'] == 1)]
    df_keep = df_keep.drop('status', axis = 1)
    df_keep = df_keep.drop('Frame_Label', axis = 1)
    
    #print('the shape of the dataframe in the status is:', df_keep.shape)
    return(df_keep)




def event_rate(df, fps):
    # 将非数值值转换为NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    
    # 获取数据框的行数和列数
    Frames, n_neuron = df.shape
    # 将数据框转换为NumPy数组
    df_array = df.to_numpy()

    peak_sum = 0
    amplitude_mean_list = []          # 用于存储每个神经元的峰值均值
    amplitude_mean_list_wit_zero = []  # 用于存储每个神经元的峰值均值（包括没有峰值的情况）
    peak_info_list = []                # 用于存储每个神经元的峰值信息

    # 遍历每个神经元
    for i in range(n_neuron):
        # 使用find_peaks函数查找峰值
        peaks, _ = find_peaks(df_array[:, i], prominence=5, width=2)   ############## find peaks ##########################################################
        peak_sum += len(peaks)  # 累计所有神经元的峰值数量
        
        # 处理有峰值的情况
        if len(peaks) != 0:
            amplitude_mean = np.mean(df_array[:, i][peaks])  # 计算有峰值神经元的峰值均值
            amplitude_mean_list.append(amplitude_mean)
            amplitude_mean_list_wit_zero.append(amplitude_mean)
        else:
            amplitude_mean_list_wit_zero.append(0)  # 没有峰值的神经元，均值为0

        # 将峰值信息添加到列表中
        peak_info_list.append({'neuron_index': i, 'peaks': peaks})

    # 计算事件率
    if Frames != 0 and n_neuron != 0:
        event_rate = peak_sum / (Frames / fps) / n_neuron
    else:
        event_rate = np.nan  # 若数据为空，事件率为NaN

    # 计算所有神经元的峰值均值
    amplitude = np.mean(amplitude_mean_list)
    # 计算所有神经元的峰值均值（包括没有峰值的情况）
    amplitude_00 = np.mean(amplitude_mean_list_wit_zero)

    # 返回计算结果
    return event_rate, amplitude, amplitude_00, df_array, peak_info_list

# 先独立运行完seperate session,再进行后面的Transient rate的分析
只需要参考一只小鼠的处理流程。不同小鼠除了文件保存路径不同外其他代码相同。

# Separate sessions & Save for later use

In [18]:
root_path = 'F:/AAA-RXC'
data_path = os.path.join(root_path)
# F:/AAA-Social analysis/Data-Neuron Social/AAA-PL/1-PL04-SOCIAL

IL1_list = ['PL1-4sessions',]
WT_mouse = ['WT']*len(IL1_list)

session_ids = [1,2,3,4]

In [19]:
#分离不同的session。根据TRACE文件中的'time'列中的值，将不同的session区分开。
def separate_session(df, threshold=3):
    df = pd.read_csv(df, index_col=0)
    
    df.columns = [c.strip() for c in df.columns]  #去除TRACE文件标题列的空格

    df['index_col'] = pd.to_numeric(df.index, errors='coerce')
    check_points = [0]  # 初始化第一个分割点为0（DataFrame的起始位置）
    frame_list = df['index_col'].values.tolist()

    # 判断两行之间的时间差，大于threshold则视为新的session
    for i in range(len(frame_list) - 1):
        if (frame_list[i+1] - frame_list[i]) > threshold:
            check_points.append(frame_list[i+1])
    
    check_points.append(frame_list[-1] + 1)  # 添加最后一个分割点

    session_dfs = []
    for i in range(len(check_points) - 1):
        session_df = df[(df['index_col'] >= check_points[i]) & (df['index_col'] < check_points[i+1])]
        session_df.drop(columns='index_col', inplace=True)
        session_dfs.append(session_df)

    return session_dfs

In [20]:
for i in range(len(IL1_list)):
    
    mouseSummary = IL1_list[i]
    mouseName = mouseSummary[-11:-7]   ### mouse name selected from file name
    mouseType = WT_mouse[i]
    print("Processing ", mouseName+'-'+mouseType)
    
    trace = os.path.join(data_path, mouseSummary, "TRACE_Con.csv")
    df_traces = separate_session(trace)

    for session_id in session_ids:
        event = os.path.join(data_path, mouseSummary, f"event{session_id}.xlsx")
        #event = os.path.join(data_path, mouseSummary, f"event.xlsx")
        #onoff = os.path.join(data_path, mouseSummary, project+'ONOFF.csv')
    
        session = df_traces[session_id-1]
        session.index.name="Frame"
        session.index = pd.to_numeric(session.index)
        session_frame_label = add_event_label(session, event)

        session_frame_label.to_csv(os.path.join(data_path, mouseSummary, f"session_{session_id}_trace.csv"))

        print(mouseName, f"session_{session_id}_trace has shape:", session_frame_label.shape)

    print('Done')
    session_frame_label.head(2)   
    

Processing  1-4s-WT


C:\Users\DELL\AppData\Local\Temp\ipykernel_46632\2922488257.py:3: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df, index_col=0)
C:\Users\DELL\AppData\Local\Temp\ipykernel_46632\2922488257.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  session_df.drop(columns='index_col', inplace=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_46632\2922488257.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.p

1-4s session_1_trace has shape: (9004, 82)
1-4s session_2_trace has shape: (6303, 82)
1-4s session_3_trace has shape: (6303, 82)
1-4s session_4_trace has shape: (6302, 82)
Done


# TransientRate_Amplitude for each mouse

In [23]:
root_path = 'F:/AAA-RXC'
data_path = os.path.join(root_path)
IL1_list = ['PL1-4sessions',]
session_ids = [1,2,]

ON_neuron_types = ['freezing_ON','freezing_ON','freezing_ON','freezing_ON',]
ON_neuron_in_which_session = 1

In [24]:
print('IL')
IL1_df_e_list = []  # event rate数据列表
IL1_df_a_list = []  # Amplitude数据列表
IL1_df_a_00_list = []  # Amplitude_With0数据列表


for i in range(len(IL1_list)):
    
    mouse_name = IL1_list[i]
    mouseSummary = IL1_list[i]
   
    print('Processing:', mouse_name)
   
    ON=ON_neuron_types[i]
    print('ON neuron interested:', ON)
    
    # 选择FPS
    if mouse_name.split('_')[0] in ['PL1-4sessions',]:
        fps = 15
    else:
        fps = 15
        
    #dfLabeled, session_1, session_2, session_3 = add_event_label(trace, event1, event2, event3)
    #session_list = [session_1, session_2, session_3]
    sessionName_list = ['session_1', 'session_2', 'session_3', 'session_4']
    

    for sessionID in session_ids:
            
            
            event = os.path.join(data_path, mouseSummary, f"event{sessionID}.xlsx")
            session = pd.read_csv(os.path.join(data_path, mouseSummary, f"session_{sessionID}_trace.csv"))
            ONOFF = pd.read_csv(os.path.join(data_path, 'Results', mouseSummary, f"session_{ON_neuron_in_which_session}_ONOFF.csv")).rename(columns={'Unnamed: 0': "Neurons"}).set_index('Neurons')
            sessionName = sessionName_list[sessionID-1]
            
            

            ###ON_list = ONOFF[ONOFF.freezing_ON==1].index.to_list()  # ###################3##########the neuron type we interested
            ON_list = ONOFF[ONOFF[ON]==1].index.to_list()   # ###################3##########the neuron type we interested
            print(mouseSummary, 'session'+str(sessionID)+' with shape:', session.shape, '| it has', len(ON_list), ON) 

            # 步骤1: 从session中提取所有神经元列名 (排除Frame和Frame_Label)
            neuron_columns = session.columns[1:-1].tolist()  # 获取C001到C136等神经元列名

            # 步骤2: 筛选匹配的列
            # 创建ON神经元编号的字符串格式列表 (带"C"前缀)
            #on_neurons_str = [f'C{str(neuron).zfill(3)}' for neuron in ON_list]

            # 找出在session中实际存在的ON神经元列
            valid_on_columns = [col for col in neuron_columns if col in ON_list]

            # 步骤3: 创建新的DataFrame (包含Frame, ON神经元和Frame_Label)
            new_session = session[["Frame"] + valid_on_columns + ["Frame_Label"]]

            # 验证结果
            print(f"原始ON_list神经元数量: {len(ON_list)}")
            print(ON_list)
            print(f"session中匹配到的ON神经元数量: {len(valid_on_columns)}")
            print(f"New session shape: {new_session.shape}")


            whole_df = new_session.drop('Frame_Label', axis=1)  
            sniff_df = check_frame_interval(new_session, 'sniff')  
            sniffed_df = check_frame_interval(new_session, 'sniffed')  
            freezing_df = check_frame_interval(new_session, 'freezing')   
            cs_df = check_frame_interval(new_session, 'cs') 
    

            # 计算各种事件率和峰值均值
            whole_eventrate, whole_amplitude, whole_amplitude_00, _, _= event_rate(whole_df, fps)
            sniff_eventrate, sniff_amplitude, sniff_amplitude_00, _, _ = event_rate(sniff_df, fps)
            sniffed_eventrate, sniffed_amplitude, sniffed_amplitude_00, _, _ = event_rate(sniffed_df, fps)
            freezing_eventrate, freezing_amplitude, freezing_amplitude_00, _, _ = event_rate(freezing_df, fps)
            cs_eventrate, cs_amplitude, cs_amplitude_00, _, _ = event_rate(cs_df, fps)

            
            # event rate
            IL1_dic_e = {'mouse_name': mouse_name,
                        'session': sessionName,
                        'whole_seesion': whole_eventrate, 
                        'sniff': sniff_eventrate, 
                        'sniffed': sniffed_eventrate,
                        'freezing/exploration': freezing_eventrate,
                        'cs/SI': cs_eventrate}
            IL1_df_e = pd.DataFrame.from_dict(IL1_dic_e, orient='index').T
            IL1_df_e_list.append(IL1_df_e)

            # Amplitude
            IL1_dic_a = {'mouse_name': mouse_name,
                        'session': sessionName,
                        'whole_seesion': whole_amplitude, 
                        'sniff': sniff_amplitude, 
                        'sniffed': sniffed_amplitude,
                        'freezing/exploration': freezing_amplitude, 
                        'cs/SI': cs_amplitude}
            IL1_df_a = pd.DataFrame.from_dict(IL1_dic_a, orient='index').T
            IL1_df_a_list.append(IL1_df_a)

            # Amplitude_With0
            IL1_dic_a_00 = {'mouse_name': mouse_name,
                        'session': sessionName,
                        'whole_seesion': whole_amplitude_00, 
                        'sniff': sniff_amplitude_00, 
                        'sniffed': sniffed_amplitude_00,
                        'freezing/exploration': freezing_amplitude_00, 
                        'cs/SI': cs_amplitude_00}
            IL1_df_a_00 = pd.DataFrame.from_dict(IL1_dic_a_00, orient='index').T
            IL1_df_a_00_list.append(IL1_df_a_00)

# 合并数据列表为数据框
IL1_df_ee = pd.concat(IL1_df_e_list)
IL1_df_aa = pd.concat(IL1_df_a_list)
IL1_df_aa_00 = pd.concat(IL1_df_a_00_list)

# 将数据框保存为CSV文件
IL1_df_ee.to_csv(f'F:/AAA-RXC/Results/Eventrate_{ON}_Across_sessions.csv')
IL1_df_aa.to_csv(f'F:/AAA-RXC/Results/Amplitude_{ON}_Across_sessions.csv')
IL1_df_aa_00.to_csv(f'F:/AAA-RXC/Results/Amplitude_With0_{ON}_Across_sessions.csv')



IL
Processing: PL1-4sessions
ON neuron interested: freezing_ON
PL1-4sessions session1 with shape: (9004, 83) | it has 24 freezing_ON
原始ON_list神经元数量: 24
['C003', 'C016', 'C017', 'C018', 'C019', 'C020', 'C022', 'C023', 'C033', 'C034', 'C035', 'C040', 'C042', 'C053', 'C054', 'C069', 'C073', 'C084', 'C097', 'C099', 'C102', 'C116', 'C125', 'C130']
session中匹配到的ON神经元数量: 24
New session shape: (9004, 26)
PL1-4sessions session2 with shape: (6303, 83) | it has 24 freezing_ON
原始ON_list神经元数量: 24
['C003', 'C016', 'C017', 'C018', 'C019', 'C020', 'C022', 'C023', 'C033', 'C034', 'C035', 'C040', 'C042', 'C053', 'C054', 'C069', 'C073', 'C084', 'C097', 'C099', 'C102', 'C116', 'C125', 'C130']
session中匹配到的ON神经元数量: 24
New session shape: (6303, 26)


C:\Users\DELL\AppData\Local\Temp\ipykernel_46632\874914387.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['status'] = check_frame_label_list
C:\Users\DELL\AppData\Local\Temp\ipykernel_46632\874914387.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['status'] = check_frame_label_list
C:\Users\DELL\AppData\Local\Temp\ipykernel_46632\874914387.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instea